# System 2 Attention
0531 用 s2a paper 的prompt實作  prompt: 給定以下新聞，提取無偏見的部分，而不是他們的意見，這樣使用該文章將是為新聞的問題部分提供無偏見答案的良好內容。

0601 更改s2a prompt 讓他用更多的偏見來實作 prompt: 給定以下新聞，提取偏見的部分，保留他們的意見，這樣使用該文章將是為新聞的問題部分提供更多偏見的內容。

In [21]:
# %pip install pandas
# %pip install numpy
# %pip install langchain_google_genai

In [22]:
import os
import textwrap
from datetime import datetime
import numpy_financial as npf
import math
import pandas as pd
import numpy as np
import json

from langchain_google_genai import ChatGoogleGenerativeAI
from google.generativeai.types import HarmBlockThreshold
from google.ai.generativelanguage_v1 import HarmCategory
from langchain_google_genai import ChatGoogleGenerativeAI

model = 'gemini-1.5-flash'
temperature = 1
top_p = 0.9
seed = 0
prompt_s2a = ""


In [23]:
def llm_response(llm, text):
    ''' 
        filepath: string 檔案路徑
        text: string 問題指令
        
        return: string 回答
    '''

    try:
        response = llm.invoke(text)
        return response.content
    except Exception as e:
        print("error" + e)
        pass
    
    return "error"

In [24]:
# llm = ChatGoogleGenerativeAI(
#                 google_api_key = os.getenv('GEMINI_API_KEY_2'),
#                 model=model,
#                 temperature=temperature,
#                 top_p=top_p,
#                 seed=seed,
#                 safety_settings={
#                         HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
#                         HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
#                         HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
#                         HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
#             })
# news = '''台積電執行副總經理暨共同營運長米玉傑在聖克拉拉（Santa Clara）的會議上表示，該技術將得以從晶圓背面向運算晶片輸送電力，有助於加快人工智慧（AI）晶片的速度。'''
# res = llm_remove_bias(llm, news)
# print(res)
# llm_response(llm, res)

In [25]:
# 轉換回答變成訊號
def response_to_signal(text):
    '''
        text: string 回答訊息
        
        return: int 回答訊號
    '''
    if text.find('no') > 0:
        signal = -1
    elif text.find('yes') > 0:
        signal = 1
    else:
        signal = 0

    return signal

# 設定要測試的股票

In [26]:
# 上市公司股票代號
# 代號 ['2618', '2881', '2880', '2211', '2731', '2707', '3023', '9941', '2308', '1514', '1101', '2542', '3036', '2385', '1303', '2912', '2323', '2603', '1419', '1604', '2486', '5269', '1314', '8033', '3017', '2903', '9943', '2368', '9904', '1102', '3231', '3004', '3035', '6176', '1103', '2609', '2002']
# 對應名稱 ['長榮航', '富邦金', '華南金', '長榮鋼', '雄獅', '晶華', '信邦', '裕融', '台達電', '亞力', '台泥', '興富發', '文曄', '群光', '南亞', '統一超', '中環', '長榮', '新光金', '聲寶', '一詮', '祥碩', '中石化', '遠雄', '奇鋐', '遠百', '好樂迪', '金像電', '寶成', '亞泥', '緯創', '豐達科', '智原', '瑞儀', '嘉泥', '陽明', '中鋼']
from datetime import datetime

start_time = datetime(2023, 1, 1)
end_time = datetime(2023, 12, 31)

stock_ids = ['9941', '2308', '1514', '1101', '2542']
stock_names = ['裕融', '台達電', '亞力', '台泥', '興富發']

In [ ]:
llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY_3'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

count = 0
for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id} {stock_names[idx]}')
    qurey = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue
        
        news = ''
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                
                with open(folder_path + file_name, 'r', encoding='utf-8') as f:
                    news = f.read()

                res_s2a = llm_response(llm, f"""<s> [INST] 給定以下新聞，提取偏見的部分，保留他們的意見，這樣使用該文章將是為新聞的問題部分提供有偏見答案的內容。[/INST] </s> 
                                    [INST] 新聞: {news} 回答: [/INST]  """)
                
                res = llm_response(llm, f"""<s> [INST] 以下是今天的新聞資訊，請對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請只回答#yes，否則只回答#no，如果與股市無關則回答#unknown，請盡量不要回答 #unknown[/INST] </s> 
                                    [INST] {res_s2a} 回答: [/INST]  """ ) 
                
                sig = response_to_signal(res)   

                count += 2
                break
            
        signal.append([date, sig]) # [日期, 訊號]
        # print(f'{date}: {sig}')

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    # ========================================================================================
    folder_path = "eval_s2a/output0601_bias/"
    # ========================================================================================
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    with open(folder_path + stock_id + "detail.json" , "w", encoding="UTF-8") as f:
        f.write(json_content)
        print("寫入檔案成功")

print(count)

# 對照組

In [30]:
# 換另一個api key
llm = ChatGoogleGenerativeAI(
                google_api_key = os.getenv('GEMINI_API_KEY_4'),
                model=model,
                temperature=temperature,
                top_p=top_p,
                seed=seed,
                safety_settings={
                        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
                        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT : HarmBlockThreshold.BLOCK_NONE,
            })

for idx, stock_id in enumerate(stock_ids):

    folder_path = os.path.dirname(os.path.abspath(os.getcwd())) + "/stock_news/" + stock_id + "/" + stock_id + "_news/"
    sorted_news_files = sorted(os.listdir(folder_path))

    # 載入價格資料
    df_price = pd.read_csv(os.path.dirname(os.path.abspath(os.getcwd())) + "/price_history/" + stock_id + "twse.csv", dtype=str)
    df_price.set_index('Date', inplace=True)  # 將'Date'行設置為索引
    
    acc_ = 0
    print(f'正在執行{stock_id} {stock_names[idx]}')
    prompt = f"請找出對{stock_id}{stock_names[idx]}的股價有影響的新聞資料"
    signal = []
    counter = 0
    for date in df_price.index:
        # 檢查日期是否在參考日期之後
        try:
            date_obj = datetime.strptime(date, '%Y%m%d')
        except:
            continue

        if not (start_time <= date_obj <= end_time):
            continue

        news = ''
        sig = 0
        for index, file_name in enumerate(sorted_news_files):
            if file_name == date:
                
                with open(folder_path + file_name, 'r', encoding='utf-8') as f:
                    news = f.read()
                
                res_s2a = llm_response(llm, f"""<s> [INST] 給定以下新聞，提取無偏見的部分，而不是他們的意見，這樣使用該文章將是為新聞的問題部分提供無偏見答案的良好內容。[/INST] </s> 
                                    [INST] 新聞: {news} 回答: [/INST]  """)

                res = llm_response(llm, f"""<s> [INST] 以下是今天的新聞資訊，請對於明天開盤時買進收盤時賣出的獲利可能性進行判斷，如果可以獲利請只回答#yes，否則只回答#no，如果與股市無關則回答#unknown，請盡量不要回答 #unknown[/INST] </s> 
                                    [INST] {res_s2a} 回答: [/INST]  """ ) 
                 
                sig = response_to_signal(res) 
                break
        
        signal.append([date, sig]) # [日期, 訊號]
        # print(f'{date}: {sig}')

    # 輸出結果 json file
    output_data = {
        "date_start": start_time.strftime("%Y%m%d"),
        "date_end": end_time.strftime("%Y%m%d"),
        "model": model,
        "temperature": temperature,
        "top_p": top_p,
        "seed": seed,
        "raw_data": signal
    }

    # 轉換成 JSON 格式
    folder_path = "eval_s2a/output0601_nobias/"
    json_content = json.dumps(output_data, ensure_ascii=False, indent=4)
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
    with open(folder_path + stock_id + "detail.json" , "w", encoding="UTF-8") as f:
        f.write(json_content)
        print("寫入檔案成功")

正在執行9941 裕融


TypeError: can only concatenate str (not "ChatGoogleGenerativeAIError") to str